In [1]:
%pip install scikit-learn pandas numpy xgboost
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\carso\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [2]:
pipes = pd.read_csv("Water_Pipes_Feb10.csv", encoding='cp1252', dtype={10: str})
breaks = pd.read_csv("BreaksWithHistorical.csv")
breaks["break_date"] = pd.to_datetime(breaks["break_date"], errors="coerce")
pipes['Soil_Hydro_Group'] = pipes['Soil_Hydro_Group'].fillna("No Group")
pipes = pipes.dropna()
pipes.isna().sum()

OBJECTID            0
tag                 0
status              0
cbu                 0
owner               0
metered             0
fire_line           0
diameter            0
material            0
polywrap            0
yr_inst             0
length              0
yr_abandon          0
yr_removed          0
yr_replace          0
comments            0
NumberOfMa          0
Shape_Leng          0
Score               0
Soil_Comp           0
Soil_Comp_Full      0
Soil_Hydro_Group    0
Shape_Length        0
ElevChange          0
NumberofSLs         0
dtype: int64

In [3]:
breaks.isna().sum()

pipe_tag      0
break_date    0
dtype: int64

In [4]:
def clean_polywrap(x):
    if x == "n":
        return 0.0
    try:
        return float(x)
    except:
        return 0.0


pipes["polywrap"] = pipes["polywrap"].apply(clean_polywrap)
pipes["yr_inst"] = pd.to_numeric(pipes["yr_inst"], errors="coerce")

Snapshots

In [5]:
print(breaks['break_date'].min(), breaks['break_date'].max())

1981-01-05 00:00:00 2025-08-15 00:00:00


In [14]:
SNAPSHOT_START = "1980-12-01"
SNAPSHOT_END   = "2025-09-01"

snapshots = pd.date_range(
    SNAPSHOT_START,
    SNAPSHOT_END,
    freq="MS"
)

breaks_by_pipe = (
    breaks
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(list)
    .to_dict()
)

rows = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    pipe_breaks = breaks_by_pipe.get(tag, [])

    for snap in snapshots:

        if snap.year < install_year:
            continue

        past = [d for d in pipe_breaks if d < snap]

        if len(past) == 0:
            months_since_last = 0
            breaks_so_far = 0
            has_broken_before = 0
        else:
            last_break = max(past)
            months_since_last = (snap - last_break).days / 30.44
            breaks_so_far = len(past)
            has_broken_before = 1

        rows.append({
            "tag": tag,
            "snapshot": snap,

            "diameter": pipe["diameter"],
            "material": pipe["material"],
            "polywrap": pipe["polywrap"],
            "yr_inst": pipe["yr_inst"],
            "length": pipe["length"],
            "Shape_Length": pipe["Shape_Length"],
            "Soil_Comp": pipe["Soil_Comp"],
            "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
            "ElevChange": pipe["ElevChange"],
            "NumberofSLs": pipe["NumberofSLs"],

            "age_years": snap.year - pipe["yr_inst"],

            "breaks_so_far": breaks_so_far,
            "months_since_last": months_since_last,
            "has_broken_before": has_broken_before
        })

snap_df = pd.DataFrame(rows)

Preprocessesing

In [16]:
breaks_by_tag = (
    breaks
    .dropna(subset=["pipe_tag", "break_date"])
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(np.array)
)

def broke_in_next(tag, snap, months):
    if tag not in breaks_by_tag:
        return 0

    dates = breaks_by_tag[tag]
    start = snap
    end   = snap + pd.DateOffset(months=months)
    start = np.datetime64(start)
    end   = np.datetime64(end)
    i = np.searchsorted(dates, start, side="right")

    if i >= len(dates):
        return 0

    return int(dates[i] <= end)

X = 24 #months

snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next(r["tag"], r["snapshot"], X),
    axis=1
)

XGBoost

In [17]:
feature_cols = [
    "diameter",
    "material",
    #"polywrap",
    "yr_inst",
    "length",
    "Shape_Length",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    "age_years",
    "breaks_so_far",
    "months_since_last",
    "has_broken_before"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2012-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [18]:
snap_df.isna().sum()

tag                      0
snapshot                 0
diameter                 0
material                 0
polywrap             12096
yr_inst                  0
length                   0
Shape_Length             0
Soil_Comp                0
Soil_Hydro_Group         0
ElevChange               0
NumberofSLs              0
age_years                0
breaks_so_far            0
months_since_last        0
has_broken_before        0
Y                        0
dtype: int64

In [43]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1500,
    tree_method="hist",
    max_depth = None,
    #gamma = 1,
    #reg_lambda = 10,
    subsample=0.8,
    colsample_bytree=0.8,
    learning_rate = 0.1
)
xgb.fit(X_train_final, y_train)
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

In [44]:
#xgb.feature_importances_
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = xgb.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
9,material,0.468316
3,Soil,0.302928
11,yr_inst,0.050219
4,age_years,0.048950
8,length,0.034550
0,ElevChange,0.030341
1,NumberofSLs,0.027480
2,Shape_Length,0.021773
6,diameter,0.015444
5,breaks_so_far,0.000000


In [45]:
print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))
print("XGB Log Loss", log_loss(y_test, p_test_xgb))
print("XGB MAE", mean_absolute_error(y_test, p_test_xgb))

XGB ROC AUC: 0.6650608012150847
XGB PR AUC : 0.019280777494707725
XGB Log Loss 0.10098508513364682
XGB MAE 0.011388243176043034


random forest

In [46]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced"
)
rf.fit(X_train_final, y_train)
p_test_rf = rf.predict_proba(X_test_final)[:, 1]

In [47]:
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = rf.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
4,age_years,0.231955
3,Soil,0.133705
11,yr_inst,0.127292
0,ElevChange,0.116536
2,Shape_Length,0.107522
9,material,0.107321
8,length,0.085119
6,diameter,0.045473
1,NumberofSLs,0.045078
5,breaks_so_far,0.000000


In [48]:
print("RF ROC AUC:", roc_auc_score(y_test, p_test_rf))
print("RF PR AUC :", average_precision_score(y_test, p_test_rf))
print("RF Log Loss", log_loss(y_test, p_test_rf))
print("RF MAE", mean_absolute_error(y_test, p_test_rf))

RF ROC AUC: 0.6733955649807496
RF PR AUC : 0.02870022198069736
RF Log Loss 0.11186944105039022
RF MAE 0.01666170248880474


OKAY, let's go back to 2013-2025 data, but lets try snaphots as seasons instead of months.

In [49]:
pipes = pd.read_csv("Water_Pipes_Feb10.csv", encoding='cp1252', dtype={10: str})
breaks = pd.read_csv("Breaks2013toNow.csv")
breaks["break_date"] = pd.to_datetime(breaks["break_date"], errors="coerce")
pipes['Soil_Hydro_Group'] = pipes['Soil_Hydro_Group'].fillna("No Group")
pipes = pipes.dropna()
pipes.isna().sum()

OBJECTID            0
tag                 0
status              0
cbu                 0
owner               0
metered             0
fire_line           0
diameter            0
material            0
polywrap            0
yr_inst             0
length              0
yr_abandon          0
yr_removed          0
yr_replace          0
comments            0
NumberOfMa          0
Shape_Leng          0
Score               0
Soil_Comp           0
Soil_Comp_Full      0
Soil_Hydro_Group    0
Shape_Length        0
ElevChange          0
NumberofSLs         0
dtype: int64

In [51]:
def clean_polywrap(x):
    if x == "n":
        return 0.0
    try:
        return float(x)
    except:
        return np.nan


pipes["polywrap"] = pipes["polywrap"].apply(clean_polywrap)
pipes["yr_inst"] = pd.to_numeric(pipes["yr_inst"], errors="coerce")

In [52]:
SNAPSHOT_START = "2013-01-01"
SNAPSHOT_END   = "2025-09-01"

snapshots = pd.date_range(
    SNAPSHOT_START,
    SNAPSHOT_END,
    freq="QS"
)
rows = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ].sort_values()

    for snap in snapshots:

        if snap.year < install_year:
            continue

        past = pipe_breaks[pipe_breaks < snap]

        if len(past) == 0:
            months_since_last = np.nan
            breaks_so_far = 0
        else:
            last_break = past.max()
            months_since_last = (snap - last_break).days / 30.44
            breaks_so_far = len(past)

        rows.append({
            "tag": tag,
            "snapshot": snap,

            "diameter": pipe["diameter"],
            "material": pipe["material"],
            "polywrap": pipe["polywrap"],
            "yr_inst": pipe["yr_inst"],
            "length": pipe["length"],
            "Shape_Length": pipe["Shape_Length"],
            "Soil_Comp": pipe["Soil_Comp"],
            "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
            "ElevChange": pipe["ElevChange"],
            "NumberofSLs": pipe["NumberofSLs"],
            "age_years": snap.year - pipe["yr_inst"],
            "breaks_so_far": breaks_so_far,
            "months_since_last": months_since_last
        })

snap_df = pd.DataFrame(rows)

In [53]:
breaks_by_tag = (
    breaks
    .dropna(subset=["pipe_tag", "break_date"])
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(np.array)
)

def broke_in_next(tag, snap, months):
    if tag not in breaks_by_tag:
        return 0

    dates = breaks_by_tag[tag]
    start = snap
    end   = snap + pd.DateOffset(months=months)
    start = np.datetime64(start)
    end   = np.datetime64(end)
    i = np.searchsorted(dates, start, side="right")

    if i >= len(dates):
        return 0

    return int(dates[i] <= end)

X = 24 #months
    
snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next(r["tag"], r["snapshot"], X),
    axis=1
)

In [87]:
feature_cols = [
    "diameter", #c
    "material", #c
    #"polywrap",
    "yr_inst", #c
    "length",
    "Shape_Length",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    "age_years",
    "breaks_so_far",  #c
    #"months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [62]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'learning_rate': [0.1, 0.01, 0.001, 0.0001], #dont need to test really high (max 0.01 starting from 0.001)
    'max_depth': [3, 5, 10, 15, 999], #most important learning_rate and max_depth also 999
    'subsample': [0.8, 1], #start with 1
    'colsample_bytree': [0.8, 1], # start with 1
    #'gamma': [0, 1, 5],
    'reg_lambda': [0, 5, 10]
}


xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1500,
    tree_method="hist",
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=1
)

grid_search.fit(X_train_final, y_train)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")
#model.get_score()
#for RF, feature selection in scikit-learn (model.feature_importances_)

Fitting 3 folds for each of 240 candidates, totalling 720 fits
Best Score: 0.03809694012479097
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'reg_lambda': 5, 'subsample': 1}


In [90]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1500,
    tree_method="hist",
    max_depth = 20,
    #gamma = 1,
    reg_lambda = 15,
    #subsample=1,
    #colsample_bytree=0.8,
    learning_rate = 0.1
)
xgb.fit(X_train_final, y_train)
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

In [85]:
#xgb.feature_importances_
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = xgb.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
3,Soil,0.679893
7,material,0.227826
4,breaks_so_far,0.016157
2,Shape_Length,0.015269
8,yr_inst,0.013363
5,diameter,0.012991
1,NumberofSLs,0.011979
6,length,0.011459
0,ElevChange,0.011063


In [91]:
print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))
print("XGB Log Loss", log_loss(y_test, p_test_xgb))
print("XGB MAE", mean_absolute_error(y_test, p_test_xgb))

XGB ROC AUC: 0.8322780447567424
XGB PR AUC : 0.2708546950294293
XGB Log Loss 0.05764961660073364
XGB MAE 0.009249648079276085


Lets see if we can improve our XGBoost model any more?

In [6]:
#print(breaks['break_date'].min(), breaks['break_date'].max())
SNAPSHOT_START = "2013-01-01"
SNAPSHOT_END   = "2025-09-01"

snapshots = pd.date_range(
    SNAPSHOT_START,
    SNAPSHOT_END,
    freq="MS"
)
rows = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ].sort_values()

    for snap in snapshots:

        if snap.year < install_year:
            continue

        past = pipe_breaks[pipe_breaks < snap]

        if len(past) == 0:
            months_since_last = np.nan
            breaks_so_far = 0
        else:
            last_break = past.max()
            months_since_last = (snap - last_break).days / 30.44
            breaks_so_far = len(past)

        rows.append({
            "tag": tag,
            "snapshot": snap,

            "diameter": pipe["diameter"],
            "material": pipe["material"],
            "polywrap": pipe["polywrap"],
            "yr_inst": pipe["yr_inst"],
            "length": pipe["length"],
            "NumberOfMa": pipe["NumberOfMa"],
            "Shape_Length": pipe["Shape_Length"],
            "Score": pipe["Score"],
            "Soil_Comp": pipe["Soil_Comp"],
            "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
            "ElevChange": pipe["ElevChange"],
            "NumberofSLs": pipe["NumberofSLs"],
            "age_years": snap.year - pipe["yr_inst"],
            "breaks_so_far": breaks_so_far,
            "months_since_last": months_since_last
        })

snap_df = pd.DataFrame(rows)

In [7]:
breaks_by_tag = (
    breaks
    .dropna(subset=["pipe_tag", "break_date"])
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(np.array)
)

def broke_in_next(tag, snap, months):
    if tag not in breaks_by_tag:
        return 0

    dates = breaks_by_tag[tag]
    start = snap
    end   = snap + pd.DateOffset(months=months)
    start = np.datetime64(start)
    end   = np.datetime64(end)
    i = np.searchsorted(dates, start, side="right")

    if i >= len(dates):
        return 0

    return int(dates[i] <= end)

#X = 24 #months

X = 60
    
snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next(r["tag"], r["snapshot"], X),
    axis=1
)

In [8]:
feature_cols = [
    #"diameter", #c
    "material", #c
    #"polywrap",
    "yr_inst", #c
    "length",
    "NumberOfMa",
    #"Shape_Length",
    "Score",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    #"age_years",
    "breaks_so_far",  #c
    #"months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [9]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1500,
    tree_method="hist",
    max_depth = 20,
    #gamma = 0.0001,
    reg_lambda = 30,
    #subsample=0.8,
    #colsample_bytree=0.8,
    learning_rate = 1,
    alpha=0.001,
    eta = 1,
    scale_pos_weight = 5
)
xgb.fit(X_train_final, y_train)
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

In [10]:
#xgb.feature_importances_
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = xgb.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
4,Soil,0.806091
7,material,0.131355
1,NumberOfMa,0.048840
5,breaks_so_far,0.004289
6,length,0.003188
0,ElevChange,0.001749
3,Score,0.001687
2,NumberofSLs,0.001447
8,yr_inst,0.001353


In [11]:
print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))
print("XGB Log Loss", log_loss(y_test, p_test_xgb))
print("XGB MAE", mean_absolute_error(y_test, p_test_xgb))

XGB ROC AUC: 0.9948761748220936
XGB PR AUC : 0.8846527805102956
XGB Log Loss 0.009490059649533167
XGB MAE 0.004577675834298134


predictions

In [11]:
today = pd.Timestamp.today().normalize()

rows_today = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year) or today.year < install_year:
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ]

    past = pipe_breaks[pipe_breaks < today]

    breaks_so_far = len(past)

    rows_today.append({
        "tag": tag,

        # model features
        "material": pipe["material"],
        "yr_inst": pipe["yr_inst"],
        "length": pipe["length"],
        "NumberOfMa": pipe["NumberOfMa"],
        "Score": pipe["Score"],
        "Soil_Comp": pipe["Soil_Comp"],
        "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
        "ElevChange": pipe["ElevChange"],
        "NumberofSLs": pipe["NumberofSLs"],
        "breaks_so_far": breaks_so_far
    })

today_df = pd.DataFrame(rows_today)

In [13]:
X_today_num = num_imp.transform(today_df[num_features])
X_today_cat = ohe.transform(today_df[cat_features])
X_today_final = np.hstack([X_today_num, X_today_cat])

In [14]:
today_df["prob_xgb"] = xgb.predict_proba(X_today_final)[:, 1]

In [17]:
#make it a csv
output = today_df[[
    "tag",
    "prob_xgb",
]].sort_values("prob_xgb", ascending=False)

output.to_csv("pipe_break_risk_report4.csv", index=False)